# Dhartisense: plant disease model v3 (Colab)

Same data and labels as v2 (PlantVillage + PlantDoc, 38 classes), with these changes aimed at real accuracy:

- **Stronger backbone:** ConvNeXt-Tiny (28M parameters, ImageNet-pretrained) instead of MobileNetV3. Small field datasets gain the most from a better pretrained backbone.
- **Higher resolution (288 px)**, so small lesions stay visible.
- **Training matches serving exactly:** images keep their aspect ratio (v2 squashed them to squares).
- **Flip averaging at test time.**
- **Near-duplicate check:** PlantDoc contains near-identical web images. Duplicates of training images are removed from the validation split (so calibration is honest), and the test score is also reported on the non-duplicate subset.
- **Two thresholds:** one for "crop unknown" and one for "farmer picked the crop" (the app's normal mode).
- **Crop-conditioned test accuracy** is reported, because that is what farmers actually get.

**Before running:** Runtime > Change runtime type > **T4 GPU**, then Run all (about 25 to 35 minutes).

**Outputs:** `plant_disease_v3.pth` and `plant_disease_v3.metrics.json`. Put both in `ml/models/` (the server prefers v3 when present) and send me the metrics file.

In [ ]:
!pip -q install kagglehub
import kagglehub, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (training will be slow)')
DATA_PD = kagglehub.dataset_download('nirmalsankalana/plantdoc-dataset')
DATA_PV = kagglehub.dataset_download('abdallahalidev/plantvillage-dataset')
print(DATA_PD); print(DATA_PV)

In [ ]:
# ---- Unified labels: every class becomes "Crop|Condition" so PlantVillage and PlantDoc merge ----
PD_LABELS = {  # PlantDoc folder -> (crop, condition)
    "Apple_leaf": ("Apple", "Healthy"), "Apple_rust_leaf": ("Apple", "Cedar apple rust"), "Apple_Scab_Leaf": ("Apple", "Apple scab"),
    "Bell_pepper_leaf": ("Bell pepper", "Healthy"), "Bell_pepper_leaf_spot": ("Bell pepper", "Bacterial leaf spot"),
    "Blueberry_leaf": ("Blueberry", "Healthy"), "Cherry_leaf": ("Cherry", "Healthy"),
    "Corn_Gray_leaf_spot": ("Corn", "Gray leaf spot"), "Corn_leaf_blight": ("Corn", "Northern leaf blight"), "Corn_rust_leaf": ("Corn", "Common rust"),
    "grape_leaf": ("Grape", "Healthy"), "grape_leaf_black_rot": ("Grape", "Black rot"), "Peach_leaf": ("Peach", "Healthy"),
    "Potato_leaf_early_blight": ("Potato", "Early blight"), "Potato_leaf_late_blight": ("Potato", "Late blight"),
    "Raspberry_leaf": ("Raspberry", "Healthy"), "Soyabean_leaf": ("Soybean", "Healthy"),
    "Squash_Powdery_mildew_leaf": ("Squash", "Powdery mildew"), "Strawberry_leaf": ("Strawberry", "Healthy"),
    "Tomato_Early_blight_leaf": ("Tomato", "Early blight"), "Tomato_leaf": ("Tomato", "Healthy"),
    "Tomato_leaf_bacterial_spot": ("Tomato", "Bacterial spot"), "Tomato_leaf_late_blight": ("Tomato", "Late blight"),
    "Tomato_leaf_mosaic_virus": ("Tomato", "Mosaic virus"), "Tomato_leaf_yellow_virus": ("Tomato", "Yellow leaf curl virus"),
    "Tomato_mold_leaf": ("Tomato", "Leaf mold"), "Tomato_Septoria_leaf_spot": ("Tomato", "Septoria leaf spot"),
    "Tomato_two_spotted_spider_mites_leaf": ("Tomato", "Spider mites"),
}
CROP_FIX = {"corn (maize)": "Corn", "cherry (including sour)": "Cherry", "pepper, bell": "Bell pepper", "pepper bell": "Bell pepper", "soyabean": "Soybean"}
COND_FIX = {  # spelling differences between the two datasets -> one name
    "cercospora leaf spot gray leaf spot": "gray leaf spot", "bacterial leaf spot": "bacterial spot",
    "two-spotted spider mite": "spider mites", "spider mites two-spotted spider mite": "spider mites",
    "haunglongbing (citrus greening)": "citrus greening", "leaf blight (isariopsis leaf spot)": "leaf blight",
}

def canon(crop, condition):
    """Return 'Crop|Condition' with the same spelling no matter which dataset the name came from."""
    crop = crop.strip().replace("_", " ")
    crop = CROP_FIX.get(crop.lower(), crop[:1].upper() + crop[1:])
    cond = condition.replace("_", " ").strip().lower()
    for prefix in (crop.lower() + " ", crop.lower().split()[0] + " "):  # PlantVillage repeats the crop in the condition
        if cond.startswith(prefix) and cond != prefix.strip():
            cond = cond[len(prefix):]
            break
    cond = COND_FIX.get(cond, cond)
    return f"{crop}|{cond.capitalize()}"

def pd_label(folder):
    return canon(*PD_LABELS[folder])

def pv_label(folder):
    crop, _, cond = folder.partition("___")
    return canon(crop, cond)

In [ ]:
import random
from collections import Counter
from pathlib import Path
IMG = ('.jpg', '.jpeg', '.png')
PV_CAP = 100          # max PlantVillage training images per class, so lab photos do not drown field photos
VAL_FRACTION = 0.15
random.seed(42)

# PlantDoc: dataset's own train/test; a validation slice of train is held out for epoch choice + calibration.
def pd_items(split):
    return [(p, pd_label(p.parent.name)) for p in sorted((Path(DATA_PD) / split).glob('*/*')) if p.suffix.lower() in IMG]
pd_train, pd_test = pd_items('train'), pd_items('test')
random.shuffle(pd_train)
cut = int(len(pd_train) * VAL_FRACTION)
pd_val, pd_train = pd_train[:cut], pd_train[cut:]

# PlantVillage: use the colour images only (not the grayscale/segmented copies some mirrors include).
pv_all = [p for p in Path(DATA_PV).rglob('*') if p.suffix.lower() in IMG and '___' in p.parent.name]
if any('color' in (x.lower() for x in p.parts) for p in pv_all):
    pv_all = [p for p in pv_all if 'color' in (x.lower() for x in p.parts)]
by_class = {}
for p in pv_all: by_class.setdefault(pv_label(p.parent.name), []).append(p)
pv_train = []
for label, paths in by_class.items():
    random.shuffle(paths)
    pv_train += [(p, label) for p in paths[:PV_CAP]]

train = pd_train + pv_train
labels = sorted({l for _, l in train + pd_val + pd_test})
ids = {l: i for i, l in enumerate(labels)}
n_pd, n_pv, n_val, n_test = Counter(l for _, l in pd_train), Counter(l for _, l in pv_train), Counter(l for _, l in pd_val), Counter(l for _, l in pd_test)
print(f'{len(labels)} classes | train={len(train)} (PlantDoc {len(pd_train)} + PlantVillage {len(pv_train)}) | val={len(pd_val)} | test={len(pd_test)}')
print(f"{'class':45s} {'PDtrain':>7s} {'PVtrain':>7s} {'val':>4s} {'test':>4s}")
for l in labels: print(f'{l:45s} {n_pd[l]:7d} {n_pv[l]:7d} {n_val[l]:4d} {n_test[l]:4d}')
assert all(n_pd[l] + n_pv[l] >= 20 for l in labels), 'a class has fewer than 20 training images'

In [ ]:
from PIL import Image
SIZE = 288        # model input size
PRELOAD = 384     # keep this much resolution in memory for crops; never upscale

def load(items):
    out = []
    for p, l in items:
        im = Image.open(p).convert('RGB'); w, h = im.size; s = PRELOAD / min(w, h)
        if s < 1: im = im.resize((round(w * s), round(h * s)), Image.BICUBIC)
        out.append((im, ids[l]))
    return out
train_data, val_data, test_data = load(train), load(pd_val), load(pd_test)

def dhash(im, n=8):  # tiny perceptual hash: near-identical images differ in <= a few bits
    px = list(im.convert('L').resize((n + 1, n), Image.LANCZOS).getdata()); bits = 0
    for r in range(n):
        for c in range(n):
            bits = (bits << 1) | (px[r * (n + 1) + c] > px[r * (n + 1) + c + 1])
    return bits
train_hashes = [dhash(im) for im, _ in train_data]
def near_duplicate(im): h = dhash(im); return any(bin(h ^ t).count('1') <= 5 for t in train_hashes)
val_dup = [near_duplicate(im) for im, _ in val_data]
test_dup = [near_duplicate(im) for im, _ in test_data]
print(f'near-duplicates of training images: val {sum(val_dup)}/{len(val_dup)}, test {sum(test_dup)}/{len(test_dup)}')
val_data = [d for d, dup in zip(val_data, val_dup) if not dup]   # calibrate only on genuinely unseen photos
print(f'validation kept: {len(val_data)}')

In [ ]:
import copy, json
from datetime import datetime, timezone
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import ConvNeXt_Tiny_Weights, convnext_tiny

EPOCHS = 25
device = 'cuda' if torch.cuda.is_available() else 'cpu'
amp = dict(device_type=device, dtype=torch.float16, enabled=device == 'cuda')
torch.manual_seed(42)
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
aug = transforms.Compose([transforms.RandomResizedCrop(SIZE, scale=(0.35, 1.0)), transforms.RandomHorizontalFlip(),
                          transforms.RandomRotation(25), transforms.ColorJitter(0.35, 0.35, 0.35, 0.05), transforms.ToTensor(), norm])
plain = transforms.Compose([transforms.Resize(int(SIZE * 1.14)), transforms.CenterCrop(SIZE), transforms.ToTensor(), norm])  # identical to server.py

class Imgs(Dataset):
    def __init__(self, data, tf): self.data, self.tf = data, tf
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        img, y = self.data[i]
        return self.tf(img), y

train_dl = DataLoader(Imgs(train_data, aug), batch_size=32, shuffle=True, num_workers=2)
val_dl = DataLoader(Imgs(val_data, plain), batch_size=64, num_workers=2)
test_dl = DataLoader(Imgs(test_data, plain), batch_size=64, num_workers=2)

per_class = Counter(y for _, y in train_data)
class_w = torch.tensor([len(train_data) / (len(labels) * max(1, per_class[c])) for c in range(len(labels))], dtype=torch.float).to(device)

model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
model.classifier[2] = nn.Linear(model.classifier[2].in_features, len(labels))
model.to(device)
head = list(model.classifier[2].parameters()); head_ids = {id(p) for p in head}
backbone = [p for p in model.parameters() if id(p) not in head_ids]
criterion = nn.CrossEntropyLoss(weight=class_w, label_smoothing=0.1)
optimizer = torch.optim.AdamW([{'params': backbone, 'lr': 1e-4}, {'params': head, 'lr': 1e-3}], weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=device == 'cuda')

def collect_logits(dl):  # flip averaging
    model.eval(); out, ys = [], []
    with torch.no_grad():
        for x, y in dl:
            x = x.to(device)
            with torch.autocast(**amp):
                lg = (model(x) + model(x.flip(-1))) / 2
            out.append(lg.float().cpu()); ys.append(y)
    return torch.cat(out), torch.cat(ys)

best_acc, best_state = -1.0, None
for epoch in range(EPOCHS):
    model.train(); total = 0.0
    for x, y in train_dl:
        optimizer.zero_grad()
        with torch.autocast(**amp):
            loss = criterion(model(x.to(device)), y.to(device))
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); total += float(loss)
    scheduler.step()
    lg, ys = collect_logits(val_dl)
    val_acc = float((lg.argmax(1) == ys).float().mean())  # field-photo validation only
    print(f'epoch {epoch + 1}/{EPOCHS} loss={total:.2f} field_val_acc={val_acc:.3f}', flush=True)
    if val_acc > best_acc:
        best_acc, best_state = val_acc, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state)

In [ ]:
import torch.nn.functional as F
val_logits, val_y = collect_logits(val_dl)
test_logits, test_y = collect_logits(test_dl)
crop_of = [l.split('|')[0] for l in labels]

def only_true_crop(logits, ys):  # the farmer picked the right crop: keep just that crop's classes
    keep = torch.tensor([[crop_of[c] == crop_of[y] for c in range(len(labels))] for y in ys.tolist()])
    return logits.masked_fill(~keep, float('-inf'))

def ece(probs, y, bins=10):
    conf, pred = probs.max(1); right = (pred == y).float(); total = 0.0
    for lo in torch.linspace(0, 1, bins + 1)[:-1]:
        m = (conf > lo) & (conf <= lo + 1 / bins)
        if m.any(): total += float(m.float().mean()) * abs(float(right[m].mean()) - float(conf[m].mean()))
    return total

TARGET = 0.90
def pick_threshold(lg, y, T):  # smallest confidence whose accepted predictions reach TARGET accuracy (min 20 accepted)
    conf, pred = F.softmax(lg / T, 1).max(1); right = (pred == y).float()
    for t in sorted(set(round(float(c), 3) for c in conf)):
        m = conf >= t
        if m.sum() >= 20 and float(right[m].mean()) >= TARGET: return t
    print('WARNING: no threshold reached the target accuracy; using 0.6'); return 0.6

temps = [round(0.5 + 0.01 * i, 2) for i in range(0, 451)]
T = min(temps, key=lambda t: float(F.cross_entropy(val_logits / t, val_y)))
threshold = pick_threshold(val_logits, val_y, T)
threshold_crop = pick_threshold(only_true_crop(val_logits, val_y), val_y, T)
print(f'temperature={T} threshold={threshold} threshold_crop={threshold_crop} | val ECE {ece(F.softmax(val_logits, 1), val_y):.3f} -> {ece(F.softmax(val_logits / T, 1), val_y):.3f}')

def score(logits, y, thr, mask=None):
    p = F.softmax(logits / T, 1); conf, pred = p.max(1)
    keep = torch.ones(len(y), dtype=torch.bool) if mask is None else mask
    acc = float((pred[keep] == y[keep]).float().mean())
    top3 = float((p[keep].topk(3, 1).indices == y[keep][:, None]).any(1).float().mean())
    acc_mask = keep & (conf >= thr)
    return dict(top1=acc, top3=top3, coverage=float((conf[keep] >= thr).float().mean()),
                accuracy_when_accepted=float((pred[acc_mask] == y[acc_mask]).float().mean()) if acc_mask.any() else None,
                ece=ece(p[keep], y[keep]))
fresh = torch.tensor([not d for d in test_dup])
results = {
    'crop_unknown': score(test_logits, test_y, threshold),
    'crop_unknown_excluding_near_duplicates': score(test_logits, test_y, threshold, fresh),
    'farmer_picks_crop': score(only_true_crop(test_logits, test_y), test_y, threshold_crop),
    'farmer_picks_crop_excluding_near_duplicates': score(only_true_crop(test_logits, test_y), test_y, threshold_crop, fresh),
}
crop_right = float(torch.tensor([crop_of[p] == crop_of[t] for p, t in zip(test_logits.argmax(1).tolist(), test_y.tolist())]).float().mean())
for name, r in results.items(): print(f'{name:48s}', {k: (round(v, 3) if v is not None else None) for k, v in r.items()})
print('crop identified correctly:', round(crop_right, 3))

pred_all = test_logits.argmax(1)
per_class_acc = {l: round(float((pred_all[test_y == i] == i).float().mean()), 3) for i, l in enumerate(labels) if (test_y == i).any()}
pv_only = [l for l in labels if not any(y == labels.index(l) for _, y in val_data + test_data)]
print('classes with NO field-photo validation/test images (confidence unverified):', pv_only)

In [ ]:
torch.save({'state_dict': best_state, 'labels': labels, 'model': 'convnext_tiny', 'image_size': SIZE, 'tta': True,
            'temperature': T, 'threshold': threshold, 'threshold_crop': threshold_crop}, 'plant_disease_v3.pth')
json.dump({'trained_at': datetime.now(timezone.utc).isoformat(), 'model': 'convnext_tiny', 'image_size': SIZE,
           'datasets': ['PlantDoc (kaggle nirmalsankalana/plantdoc-dataset)', f'PlantVillage (kaggle abdallahalidev/plantvillage-dataset), max {PV_CAP} per class'],
           'classes': labels, 'epochs': EPOCHS, 'train': len(train_data), 'val': len(val_data), 'test': len(test_data),
           'test_near_duplicates_of_train': int(sum(test_dup)), 'temperature': T, 'threshold': threshold, 'threshold_crop': threshold_crop,
           'threshold_target_accuracy': TARGET, 'field_val_accuracy': best_acc, 'crop_identified_correctly': crop_right,
           'test': results, 'test_per_class_accuracy': per_class_acc, 'classes_without_field_validation': pv_only,
           'limitations': ['Test = PlantDoc test split only (field photos, 4-15 images per class): wide error bars (about +-5 points).',
                           'PlantVillage-only classes have no field-photo check; treat their confidence as unverified.',
                           'Likely-classification aid, not a confirmed diagnosis.']},
          open('plant_disease_v3.metrics.json', 'w'), indent=2)
from google.colab import files
files.download('plant_disease_v3.pth'); files.download('plant_disease_v3.metrics.json')